In [7]:
from pathlib import Path
from src.data_models import SuspectExperiment

SUSPECT_DATA_2 = SuspectExperiment.model_validate_json(
    Path("../processed/suspect_3_10.json").read_text()
)

week_labels = {
    "2026-03-02": "Week 1", "2026-03-04": "Week 1", "2026-03-06": "Week 1",
    "2026-03-09": "Week 2", "2026-03-11": "Week 2", "2026-03-13": "Week 2",
    "2026-03-16": "Week 3", "2026-03-18": "Week 3", "2026-03-20": "Week 3",
}
week_colors = {"Week 1": "tab:blue", "Week 2": "tab:orange", "Week 3": "tab:green"}

ValidationError: 9 validation errors for SuspectExperiment
runs.0.per_second_data
  Field required [type=missing, input_value={'run_id': 's3_run_2026-0...0', 'heart_rate': 110}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
runs.1.per_second_data
  Field required [type=missing, input_value={'run_id': 's3_run_2026-0...0', 'heart_rate': 117}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
runs.2.per_second_data
  Field required [type=missing, input_value={'run_id': 's3_run_2026-0...0', 'heart_rate': 109}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
runs.3.per_second_data
  Field required [type=missing, input_value={'run_id': 's3_run_2026-0...0', 'heart_rate': 125}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
runs.4.per_second_data
  Field required [type=missing, input_value={'run_id': 's3

In [6]:
import pandas as pd
import plotly.graph_objects as go

# Walk/run interval marks (MM:SS → seconds, label indicates phase STARTING at that time)
_marks = [
    ( "1:00", "walk"),
    ( "2:10", "run"),
    ( "3:10", "walk"),
    ( "4:30", "run"),
    ( "5:30", "walk"),
    ( "7:00", "run"),
    ( "8:00", "walk"),
    ( "9:40", "run"),
    ("10:40", "walk"),
    ("12:30", "run"),
    ("13:30", "walk"),
    ("15:30", "run"),
    ("18:30", "walk"),
]

def _mmss(s):
    m, sec = s.split(":")
    return int(m) * 60 + int(sec)

intervals = []
for i, (t, phase) in enumerate(_marks):
    t0 = _mmss(_marks[i - 1][0]) if i > 0 else 0
    t1 = _mmss(t)
    intervals.append((t0, t1, phase))

colors_map = {
    "walk": "rgba(100, 149, 237, 0.12)",
    "run":  "rgba(220,  80,  80, 0.12)",
}

trace_colors = [
    "#1f77b4", "#4a9fd4", "#7fbfea",
    "#ff7f0e", "#ffaa4d", "#ffcc80",
    "#2ca02c", "#6dd56d", "#6da56d"
]

fig = go.Figure()

for run, color in zip(SUSPECT_DATA_2.runs, trace_colors):
    date = run.run_id.replace("s2_run_", "")
    week = week_labels.get(date, "Unknown")

    timestamps = pd.Series([d.timestamp for d in run.per_second_data])
    heart_rates = [d.heart_rate for d in run.per_second_data]
    elapsed_sec = (timestamps - timestamps.iloc[0]).dt.total_seconds()

    fig.add_trace(go.Scatter(
        x=elapsed_sec,
        y=heart_rates,
        mode="lines",
        name=f"{date} ({week})",
        line=dict(color=color, width=1.5),
    ))

for t0, t1, phase in intervals:
    fig.add_vrect(
        x0=t0, x1=t1,
        fillcolor=colors_map[phase],
        layer="below",
        line_width=0,
        annotation_text=phase,
        annotation_position="top left",
        annotation_font_size=9,
        annotation_font_color="#555",
    )

fig.update_layout(
    title="Exercise Heart Rate — All Sessions Overlaid (relative time)",
    xaxis_title="Elapsed time (s)",
    yaxis_title="Heart Rate (bpm)",
    legend=dict(
        font=dict(size=11),
        title="Session",
        title_font=dict(size=12),
        orientation="h",
        yanchor="bottom",
        y=-0.2,
    ),
    height=800,
    hovermode="x unified",
)
fig.write_html("test_2.html")
fig

In [52]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_models import SuspectExperiment

SUSPECT_DATA_3 = SuspectExperiment.model_validate_json(
    Path("../processed/suspect_3.json").read_text()
)

week_labels = {
    "2026-03-02": "Week 1", "2026-03-04": "Week 1", "2026-03-06": "Week 1",
    "2026-03-09": "Week 2", "2026-03-11": "Week 2", "2026-03-13": "Week 2",
    "2026-03-16": "Week 3", "2026-03-18": "Week 3", "2026-03-20": "Week 3",
}
week_colors = {
    "Week 1": "#1f77b4",
    "Week 2": "#ff7f0e",
    "Week 3": "#2ca02c",
}

n_runs = len(SUSPECT_DATA_3.runs)

fig = make_subplots(
    rows=n_runs,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[
        f"{run.run_id.replace('s3_run_', '')} ({week_labels.get(run.run_id.replace('s3_run_', ''), '')})"
        for run in SUSPECT_DATA_3.runs
    ],
)

for i, run in enumerate(SUSPECT_DATA_3.runs, start=1):
    date = run.run_id.replace("s3_run_", "")
    week = week_labels.get(date, "Unknown")
    color = week_colors.get(week, "#888")

    timestamps = pd.Series([d.timestamp for d in run.per_second_data])
    heart_rates = [d.heart_rate for d in run.per_second_data]
    elapsed_sec = (timestamps - timestamps.iloc[0]).dt.total_seconds()

    fig.add_trace(
        go.Scatter(
            x=elapsed_sec,
            y=heart_rates,
            mode="lines",
            name=f"{date} ({week})",
            line=dict(color=color, width=1.5),
            showlegend=(i == 1 or week != week_labels.get(
                SUSPECT_DATA_3.runs[i - 2].run_id.replace("s3_run_", ""), ""
            )),
            legendgroup=week,
            customdata=timestamps.dt.strftime("%H:%M:%S").values,
            hovertemplate="Elapsed: %{x}s<br>Time: %{customdata}<br>HR: %{y} bpm<extra></extra>",
        ),
        row=i,
        col=1,
    )

    # Vertical gridlines every 60 seconds
    max_elapsed = int(elapsed_sec.max())
    for t in range(0, max_elapsed + 60, 60):
        fig.add_vline(
            x=t,
            line=dict(color="rgba(150,150,150,0.3)", width=1, dash="dot"),
            row=i,
            col=1,
        )

    fig.update_yaxes(title_text="BPM", row=i, col=1)

    # Tick labels show elapsed seconds on top, real timestamp below
    tick_every = 60
    tick_vals = list(range(0, max_elapsed + tick_every, tick_every))
    tick_texts = [
        f"{int(t)}s<br>{(timestamps.iloc[0] + pd.Timedelta(seconds=t)).strftime('%H:%M:%S')}"
        for t in tick_vals
    ]
    fig.update_xaxes(
        tickvals=tick_vals,
        ticktext=tick_texts,
        tickangle=-45,
        row=i,
        col=1,
    )

fig.update_xaxes(title_text="Elapsed time (s)", row=n_runs, col=1)

fig.update_layout(
    title="Suspect 3 — Individual Runs (elapsed time)",
    height=300 * n_runs,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.05,
        font=dict(size=11),
    ),
)

output_path = Path("suspect_3_runs.html")
fig.write_html(output_path)
print(f"Saved to {output_path}")

Saved to suspect_3_runs.html


In [5]:
import json
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

file_path = Path("/processed/suspect_4.json")
if not file_path.exists():
    print(f"Error: {file_path} not found.")
    exit()

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)


week_labels = {
    "2026-03-03": "Week 1", "2026-03-05": "Week 1", "2026-03-07": "Week 1",
    "2026-03-10": "Week 2", "2026-03-12": "Week 2", "2026-03-14": "Week 2",
    "2026-03-17": "Week 3", "2026-03-19": "Week 3", "2026-03-21": "Week 3",
}

trace_colors = [
    "#1f77b4", "#4a9fd4", "#7fbfea", # Week 1
    "#ff7f0e", "#ffaa4d", "#ffcc80", # Week 2
    "#2ca02c", "#6dd56d", "#6da56d"  # Week 3
]


TIME_LIMIT = 18 * 60 + 30 


_marks = [
    ("1:00", "walk"), ("2:10", "run"), ("3:10", "walk"), ("4:30", "run"),
    ("5:30", "walk"), ("7:00", "run"), ("8:00", "walk"), ("9:40", "run"),
    ("10:40", "walk"), ("12:30", "run"), ("13:30", "walk"), ("15:30", "run"),
    ("18:30", "walk"),
]

def _mmss(s):
    m, sec = s.split(":")
    return int(m) * 60 + int(sec)

intervals = []
for i, (t, phase) in enumerate(_marks):
    t0 = _mmss(_marks[i - 1][0]) if i > 0 else 0
    t1 = _mmss(t)
    if t0 < TIME_LIMIT:
        intervals.append((t0, min(t1, TIME_LIMIT), phase))

colors_map = {
    "walk": "rgba(100, 149, 237, 0.12)",
    "run":  "rgba(220,  80,  80, 0.12)",
}


fig = go.Figure()

for run, color in zip(data['runs'], trace_colors):
    date = run['run_id'].replace("s4_run_", "")
    week = week_labels.get(date, "Unknown")
    steps = run['metadata'].get('steps_count', 'N/A')

    if not run['per_second_data']:
        continue

    df = pd.DataFrame(run['per_second_data'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    df['elapsed'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    
    df_filtered = df[df['elapsed'] <= TIME_LIMIT]

    fig.add_trace(go.Scatter(
        x=df_filtered['elapsed'],
        y=df_filtered['heart_rate'],
        mode="lines",
        name=f"{date} ({week})",
        line=dict(color=color, width=2),
        hovertemplate=(
            f"<b>Date: {date}</b><br>" +
            f"Steps: {steps}<br>" +
            "Elapsed: %{x}s<br>" +
            "HR: %{y} bpm<extra></extra>"
        )
    ))

for t0, t1, phase in intervals:
    fig.add_vrect(
        x0=t0, x1=t1,
        fillcolor=colors_map[phase],
        layer="below",
        line_width=0,
        annotation_text=phase if t1 - t0 > 20 else "", 
        annotation_position="top left",
        annotation_font_size=10,
    )

fig.update_layout(
    title=dict(
        text="Exercise Heart Rate for Suspect 4",
        x=0.5, font=dict(size=22)
    ),
    xaxis_title="Elapsed Time (Seconds)",
    yaxis_title="Heart Rate (BPM)",
    xaxis=dict(
        range=[0, TIME_LIMIT], 
        dtick=120, 
        gridcolor='rgba(200,200,200,0.2)'
    ),
    yaxis=dict(range=[100, 210], gridcolor='rgba(200,200,200,0.2)'),
    legend=dict(orientation="h", y=-0.25, x=0.5, xanchor="center"),
    template="plotly_white",
    height=850,
    hovermode="x unified",
)


output_file = "suspect_4_runs.html"
fig.write_html(output_file)
print(f"Graph saved to: {output_file}")

Graph saved to: suspect_4_runs.html
